# Batch order PlanetScope scenes → Google Earth Engine

Reads scene IDs from the `scene_id` column of the team sheet and orders them, grouped by the `aoi` column: one clip AOI per order, and one Earth Engine collection per AOI.

**Secrets needed** (Colab key icon, notebook access on): `PL_API_KEY` and `PLANET_GEE_SA` (base64 service-account key).

Run top to bottom. `DRY_RUN = True` first: it checks everything and shows what would be ordered **without spending quota**. Then set it to `False` and re-run from step 5.

## 1. Settings

In [9]:
GEE_PROJECT = "groupprojectgeog761"
SHEET_URL = None  # or set to None and upload a CSV/XLSX in step 2
SHEET_TAB = None                            # tab name, or None for the first tab
BUNDLE = "analytic_sr_udm2"                 # 4-band SR + UDM2
BATCH = 100                                 # max scenes per order
COLLECTION_PREFIX = "planet_"               # collection per AOI: planet_<aoi file name>
DRY_RUN = False                             # True = check only, no orders placed

## 2. Setup, authentication and reading the sheet

In [2]:
!pip -q install geemap

import json, re, time
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping, shape
from google.colab import userdata, files, auth
import ee

PL_API_KEY = userdata.get("PL_API_KEY")
PLANET_GEE_SA = userdata.get("PLANET_GEE_SA")
assert PL_API_KEY, "PL_API_KEY secret missing"
assert PLANET_GEE_SA, "PLANET_GEE_SA secret missing"

session = requests.Session()
session.headers.update({"Authorization": f"api-key {PL_API_KEY}", "Content-Type": "application/json"})
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"
DATA_URL = "https://api.planet.com/data/v1/item-types/PSScene/items/{}"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

if SHEET_URL:
    auth.authenticate_user()
    import gspread, google.auth
    creds, _ = google.auth.default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(SHEET_URL)
    ws = sh.worksheet(SHEET_TAB) if SHEET_TAB else sh.sheet1
    values = ws.get_all_values()
    df = pd.DataFrame(values[1:], columns=[c.strip() for c in values[0]])
else:
    up = files.upload()
    name = next(iter(up))
    df = pd.read_excel(name, dtype=str) if name.endswith(("xlsx", "xls")) else pd.read_csv(name, dtype=str)
    df.columns = [c.strip() for c in df.columns]

print(len(df), "rows | columns:", list(df.columns))

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Saving scene_selection - Ark 1.csv to scene_selection - Ark 1.csv
30 rows | columns: ['aoi', 'scene_id', 'acquired', 'Weather condition', 'Number totally visible', 'Number possible', 'Number wakes', 'Number docked', 'Comment']


## 3. Clean and validate the scene list

In [3]:
ID_RE = re.compile(r"^\d{8}_\d{6}(_\d{2})?_[0-9a-f]{4}$")

df = df[["aoi", "scene_id"]].astype(str).apply(lambda s: s.str.strip())
df = df[(df.scene_id != "") & (df.scene_id.str.lower() != "nan")]

bad = df[~df.scene_id.str.match(ID_RE)]
if len(bad):
    print("Ignoring rows with malformed scene IDs:")
    print(bad.to_string())
df = df[df.scene_id.str.match(ID_RE)]

no_aoi = df[(df.aoi == "") | (df.aoi.str.lower() == "nan")]
if len(no_aoi):
    print(f"Ignoring {len(no_aoi)} rows with no AOI")
df = df.drop(no_aoi.index)

dups = df[df.duplicated(["aoi", "scene_id"])]
if len(dups):
    print(f"Dropping {len(dups)} duplicate rows")
df = df.drop_duplicates(["aoi", "scene_id"])

print(f"\n{len(df)} scenes across {df.aoi.nunique()} AOI(s):")
print(df.groupby("aoi").size().to_string())

Ignoring rows with malformed scene IDs:
                       aoi                scene_id
4  Nukualofa_wharf.geojson  20220405_212626_1_1005

29 scenes across 1 AOI(s):
aoi
Nukualofa_wharf.geojson    29


## 4. Upload the AOI files
Select **all** the GeoJSON files named in the `aoi` column (multi-select is fine). Any CRS works; they are reprojected to EPSG:4326 and dissolved.

In [4]:
needed = sorted(df.aoi.unique())
print("Needed:", needed)
uploaded = files.upload()

AOIS = {}
for name in needed:
    if name not in uploaded and not Path(name).exists():
        print("MISSING:", name, "- its scenes will be skipped")
        continue
    g = gpd.read_file(name).to_crs(4326)
    geom = g.union_all() if hasattr(g, "union_all") else g.unary_union
    stem = re.sub(r"[^A-Za-z0-9_-]", "_", Path(name).stem).lower()
    AOIS[name] = {"geom": geom, "geojson": mapping(geom), "collection": f"{COLLECTION_PREFIX}{stem}"}
    print(f"{name}: {geom.geom_type}, collection -> {AOIS[name]['collection']}")

df = df[df.aoi.isin(AOIS)]

Needed: ['Nukualofa_wharf.geojson']


Saving Nukualofa_wharf.geojson to Nukualofa_wharf.geojson
Nukualofa_wharf.geojson: Polygon, collection -> planet_nukualofa_wharf


## 5. Check each scene: permission, overlap, already delivered

In [10]:
def existing_scene_ids(collection_path):
    try:
        assets = ee.data.listAssets({"parent": collection_path}).get("assets", [])
    except ee.EEException:
        return set()
    return {Path(a["name"]).name[:23] for a in assets} | {Path(a["name"]).name[:20] for a in assets}

plan = {}       # aoi -> list of scene IDs to order
report = []
for aoi_name, rows in df.groupby("aoi"):
    a = AOIS[aoi_name]
    col_path = f"projects/{GEE_PROJECT}/assets/{a['collection']}"
    have = existing_scene_ids(col_path)
    plan[aoi_name] = []
    for sid in rows.scene_id:
        status = "ok"
        if sid in have:
            status = "already in collection"
        else:
            r = session.get(DATA_URL.format(sid))
            if r.status_code != 200:
                status = f"not found ({r.status_code})"
            else:
                item = r.json()
                if not any("ortho_analytic_4b_sr:download" in p for p in item.get("_permissions", [])):
                    status = "no download permission"
                elif not shape(item["geometry"]).intersects(a["geom"]):
                    status = "does not overlap AOI"
        if status == "ok":
            plan[aoi_name].append(sid)
        report.append({"aoi": aoi_name, "scene_id": sid, "status": status})

report = pd.DataFrame(report)
print(report.status.value_counts().to_string())
problems = report[report.status != "ok"]
if len(problems):
    print("\nNot ordered:")
    print(problems.to_string(index=False))
print(f"\nTo order: {sum(len(v) for v in plan.values())} scenes")

status
ok    29

To order: 29 scenes


## 6. Place the orders
Skipped while `DRY_RUN = True`.

In [11]:
ORDERS = []   # (order_id, aoi, n_scenes)

if DRY_RUN:
    print("DRY_RUN is on: nothing ordered. Set DRY_RUN = False in step 1 and re-run steps 5-6.")
else:
    for aoi_name, ids in plan.items():
        if not ids:
            continue
        a = AOIS[aoi_name]
        col_path = f"projects/{GEE_PROJECT}/assets/{a['collection']}"
        try:
            ee.data.getAsset(col_path)
        except ee.EEException:
            ee.data.createAsset({"type": "IMAGE_COLLECTION"}, col_path)
            print("Created", col_path)

        for i in range(0, len(ids), BATCH):
            chunk = ids[i:i + BATCH]
            body = {
                "name": f"{a['collection']}_{i // BATCH:02d}",
                "products": [{"item_ids": chunk, "item_type": "PSScene", "product_bundle": BUNDLE}],
                "tools": [{"clip": {"aoi": a["geojson"]}}],
                "delivery": {
                    "google_earth_engine": {
                        "project": GEE_PROJECT,
                        "collection": a["collection"],
                        "credentials": PLANET_GEE_SA,
                    }
                },
                "notifications": {"email": True},
            }
            r = session.post(ORDERS_URL, data=json.dumps(body))
            if r.status_code != 202:
                print(f"FAILED {aoi_name} batch {i // BATCH}: {r.status_code} {r.text[:300]}")
                continue
            oid = r.json()["id"]
            ORDERS.append((oid, aoi_name, len(chunk)))
            print(f"Ordered {len(chunk)} scenes for {aoi_name}: {oid}")

    pd.DataFrame(ORDERS, columns=["order_id", "aoi", "n_scenes"]).to_csv("planet_orders.csv", index=False)
    print("Order IDs saved to planet_orders.csv")

Created projects/groupprojectgeog761/assets/planet_nukualofa_wharf
Ordered 29 scenes for Nukualofa_wharf.geojson: 3e321a4b-8f9e-4eb8-b08a-d42113980265
Order IDs saved to planet_orders.csv


## 7. Wait for all orders
Polls every 2 minutes. Safe to stop and re-run; you also get an email per order.

In [12]:
pending = {oid for oid, _, _ in ORDERS}
final = {}
while pending:
    for oid in sorted(pending):
        o = session.get(f"{ORDERS_URL}/{oid}").json()
        if o["state"] in ("success", "partial", "failed", "cancelled"):
            final[oid] = o
            pending.discard(oid)
    print(time.strftime("%H:%M:%S"), f"done {len(final)}/{len(ORDERS)}")
    if pending:
        time.sleep(120)

for oid, aoi_name, n in ORDERS:
    o = final[oid]
    print(f"\n{aoi_name} | {oid} | {o['state']} | {n} scenes")
    for res in o.get("_links", {}).get("results", []):
        if res.get("delivery") != "success":
            print("   not delivered:", res.get("name"))
    for hint in o.get("error_hints", []):
        print("   hint:", hint)

00:02:46 done 0/1
00:04:47 done 0/1
00:06:47 done 0/1
00:08:47 done 0/1
00:10:47 done 0/1
00:12:47 done 0/1
00:14:47 done 0/1
00:16:48 done 0/1
00:18:48 done 1/1

Nukualofa_wharf.geojson | 3e321a4b-8f9e-4eb8-b08a-d42113980265 | success | 29 scenes


## 8. Check the collections
Ingestion lags the order by a few minutes; re-run until the counts settle.

In [13]:
for aoi_name, a in AOIS.items():
    col = ee.ImageCollection(f"projects/{GEE_PROJECT}/assets/{a['collection']}")
    try:
        n = col.size().getInfo()
    except ee.EEException:
        n = 0
    expected = (df.aoi == aoi_name).sum()
    print(f"{a['collection']}: {n} images (sheet lists {expected})")

planet_nukualofa_wharf: 29 images (sheet lists 29)


In [14]:
import ee, geemap
ee.Initialize(project="groupprojectgeog761")

col = ee.ImageCollection("projects/groupprojectgeog761/assets/planet_nukualofa_wharf")
print(col.aggregate_array("system:index").getInfo())

img = col.filter(ee.Filter.stringStartsWith("system:index", "20250110_220650_09_24dd")).first()
m = geemap.Map()
m.addLayer(img, {"bands": ["B3", "B2", "B1"], "min": 0, "max": 3000}, "True colour")
m.centerObject(img, 15)
m

['20190623_213507_1054_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20190623_213508_1054_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20190808_213145_1014_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20190821_230214_38_1069_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20200108_214853_44_105c_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20200510_224855_71_106c_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20210421_221354_54_105a_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20210722_222152_01_1065_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20210927_210254_94_2464_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20211129_210042_79_2430_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20211222_214846_03_2403_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4eb8-b08a-d42113980265', '20220114_210439_51_2223_3B_AnalyticMS_SR_clip_3e321a4b-8f9e-4